## Speaker Identification – Voice-Based Attendance System

This notebook trains a **KNN + SVM** model on MFCC features extracted from
student voice recordings, evaluates accuracy, and saves the model for use
in the standalone attendance system (`attendance.py`).

## Libraries Used

In [ ]:
import os
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import joblib


## Dataset Path (relative – works on any machine)

In [ ]:
# All paths are relative to the notebook location
DATASET_PATH = os.path.join(os.getcwd(), 'recording2.0')
SAMPLE_RATE  = 16000   # Hz – consistent throughout the project
N_MFCC       = 13      # number of MFCC coefficients

print('Dataset path:', DATASET_PATH)
print('Speakers found:', sorted(os.listdir(DATASET_PATH)))


## Testing Sample Data

In [ ]:
import IPython.display as ipd

# Use the first .wav file found in the first speaker folder
first_speaker = sorted(os.listdir(DATASET_PATH))[0]
first_folder  = os.path.join(DATASET_PATH, first_speaker)
sample_file   = next(
    os.path.join(first_folder, f)
    for f in sorted(os.listdir(first_folder)) if f.endswith('.wav')
)
print('Playing:', sample_file)
ipd.display(ipd.Audio(filename=sample_file))


## Data Visualisation

In [ ]:
audio, sr = librosa.load(sample_file, sr=SAMPLE_RATE)

plt.figure(figsize=(10, 4))
librosa.display.waveshow(audio, sr=sr)
plt.title(f'Waveform – {first_speaker}')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()


## Feature Extraction (MFCC + Deltas)

In [ ]:
def extract_features(file_path, sr=SAMPLE_RATE, n_mfcc=N_MFCC):
    """Return a fixed-length feature vector (4 × n_mfcc) for one WAV file."""
    audio, _ = librosa.load(file_path, sr=sr)
    audio, _ = librosa.effects.trim(audio)         # strip silence

    mfcc    = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
    delta   = librosa.feature.delta(mfcc)
    delta2  = librosa.feature.delta(mfcc, order=2)

    return np.hstack([
        np.mean(mfcc,   axis=1),   # mean of each MFCC
        np.std(mfcc,    axis=1),   # std  of each MFCC
        np.mean(delta,  axis=1),   # velocity
        np.mean(delta2, axis=1),   # acceleration
    ])                             # shape: (4 × n_mfcc,) = (52,)


features, labels = [], []

for person in sorted(os.listdir(DATASET_PATH)):
    person_path = os.path.join(DATASET_PATH, person)
    if not os.path.isdir(person_path):
        continue
    for fname in sorted(os.listdir(person_path)):
        if not fname.lower().endswith('.wav'):
            continue
        fpath = os.path.join(person_path, fname)
        try:
            features.append(extract_features(fpath))
            labels.append(person)
            print(f'  ✅  {person}/{fname}')
        except Exception as exc:
            print(f'  ⚠️  {person}/{fname}: {exc}')

X = np.array(features)
y = np.array(labels)

print(f'\nDataset: {X.shape[0]} samples, {X.shape[1]} features, {len(set(y))} speakers')


## Dataset Summary

In [ ]:
from collections import Counter
counts = Counter(y)
print('Samples per speaker:')
for name, n in sorted(counts.items()):
    print(f'  {name}: {n}')
print(f'\nTotal: {len(y)} samples across {len(counts)} speakers')


## Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)}  Test: {len(X_test)}')


## Model Training – KNN

In [ ]:
# Scale features before KNN (distance-based model)
knn_scaler = StandardScaler()
X_train_knn = knn_scaler.fit_transform(X_train)
X_test_knn  = knn_scaler.transform(X_test)

knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train_knn, y_train)

y_pred_knn = knn_model.predict(X_test_knn)
print('KNN Accuracy:', accuracy_score(y_test, y_pred_knn))


## KNN – Classification Report

In [ ]:
print(classification_report(y_test, y_pred_knn))


### SVM – Recommended Model

In [ ]:
svm_scaler = StandardScaler()
X_train_svm = svm_scaler.fit_transform(X_train)
X_test_svm  = svm_scaler.transform(X_test)

# probability=True enables confidence scores during inference
svm_model = SVC(kernel='rbf', C=10, gamma='scale', probability=True)
svm_model.fit(X_train_svm, y_train)

y_pred_svm = svm_model.predict(X_test_svm)
print('SVM Accuracy:', accuracy_score(y_test, y_pred_svm))


## SVM – Classification Report

In [ ]:
print(classification_report(y_test, y_pred_svm))


## Save Models for Attendance System

In [ ]:
os.makedirs('models', exist_ok=True)

joblib.dump(svm_model,  'models/svm_model.pkl')
joblib.dump(svm_scaler, 'models/svm_scaler.pkl')
joblib.dump(knn_model,  'models/knn_model.pkl')
joblib.dump(knn_scaler, 'models/knn_scaler.pkl')

print('✅ Models saved to models/')
print('   svm_model.pkl, svm_scaler.pkl  ← used by attendance.py')
print('   knn_model.pkl, knn_scaler.pkl  ← alternative model')


## Predict Helper (uses saved SVM)

In [ ]:
def predict_voice(file_path, confidence_threshold=0.5):
    """Identify the speaker from a WAV file.
    Returns (speaker_name, confidence) or ('Unknown', conf) if below threshold.
    """
    feat = extract_features(file_path).reshape(1, -1)
    feat_scaled = svm_scaler.transform(feat)

    speaker    = svm_model.predict(feat_scaled)[0]
    confidence = svm_model.predict_proba(feat_scaled).max()

    if confidence < confidence_threshold:
        return 'Unknown', confidence
    return speaker, confidence


# Quick test on a known file
speaker, conf = predict_voice(sample_file)
print(f'Predicted: {speaker}  (confidence: {conf:.2%})')
print(f'Actual   : {first_speaker}')


## Live Testing (microphone)

> Requires `sounddevice` – install via `pip install sounddevice scipy`  
> Skip this cell if running without a microphone.

In [ ]:
try:
    import sounddevice as sd
    from scipy.io.wavfile import write as wav_write

    RECORD_SECONDS = 3
    print(f'🎤 Recording for {RECORD_SECONDS}s … speak now!')
    recording = sd.rec(int(RECORD_SECONDS * SAMPLE_RATE),
                       samplerate=SAMPLE_RATE, channels=1)
    sd.wait()
    wav_write('test.wav', SAMPLE_RATE, recording)
    print('✅ Saved to test.wav')

    speaker, conf = predict_voice('test.wav')
    print(f'🎤 Detected: {speaker}  (confidence: {conf:.2%})')

except ImportError:
    print('ℹ️  sounddevice not installed – skipping live recording.')
    print('   Run: pip install sounddevice scipy')
